# Land use from LCS Meta-analysis 

THINGS TO DO:
- Land use from m2/FU to m2 --> need to multiply by production, where is the produciton in the spreadsheet?
    - Multiply land use values for arable (animal) products by the total UK production. This should result in a total land area, likely to be different from the total arable (pasture) area in the UK
- Scale all the land use factors associated with arable and animal products so the total arable and pasture land matches the totals in the UK

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../')))

from agrifoodpy_data.food import FAOSTAT

In [2]:
def vprint(list):
    """Unpack and print list items on separate lines."""
    print(*list, sep='\n')

In [3]:
# We care about items produced in the UK. Matching has already been done here:

# "Matching National Inventory Items and FAOSTAT Items >> Plant items"
gsheetkey = "1M9WcEpb1CC9VkOohmx3rjMk52ZVQWStc1OpUcNIdZ60"
url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'

cols = [
    "FAOSTAT Item Code",
    "FAOSTAT Item",
    "Total mass produced in the UK (kg, FAOSTAT 2020 from AgriFoodPy)",
    "Item to match directly from PN18"
]

matching_df_plants = pd.read_excel(
    url,
    sheet_name="Plant items",
    usecols=cols
)

matching_df_animals = pd.read_excel(
    url,
    sheet_name="Animal items",
    usecols=cols
)

matching_df_plants = matching_df_plants[matching_df_plants[cols[2]].notna()]
matching_df_animals = matching_df_animals[matching_df_animals[cols[2]].notna()]

plant_items = matching_df_plants[cols[0]].tolist()
animal_items = matching_df_animals[cols[0]].tolist()

In [4]:
ukfoodprod = FAOSTAT["production"].isel(
    Year=-1, # most recent year
).sel(
    Region=229, # United Kingdom
    Item=plant_items + animal_items
)

ukfoodprod

<xarray.DataArray 'production' (Item: 47)> Size: 188B
array([6.3200e+02, 8.1170e+03, 3.2360e+03, 8.8500e+02, 9.4000e+01,
       1.0000e+00, 2.1600e+02, 1.0000e+00, 9.0000e+00, 3.2000e+01,
       1.0310e+03, 5.2000e+01, 6.3000e+01, 3.9600e+02, 1.6000e+02,
       5.5130e+03, 6.9000e+02, 5.8100e+02, 1.0380e+03, 7.2000e+01,
       1.3400e+02, 9.0600e+02, 5.9800e+03, 4.6600e+02, 6.5000e+01,
       2.1630e+03, 9.6580e+03, 1.0000e+00, 9.3200e+02, 2.0000e+02,
       1.2820e+01, 1.0000e+01, 7.4460e+01, 2.2631e+02, 7.8500e+02,
       2.2500e+02, 1.1750e+01, 6.0000e-01, 2.0405e+02, 8.0000e+00,
       1.5229e+04, 7.9020e+01, 2.9600e+02, 2.4800e+02, 2.5014e+02,
       9.8400e+02, 1.9930e+03], dtype=float32)
Coordinates:
  * Item         (Item) int64 376B 2617 2513 2656 2658 ... 2736 2763 2733 2734
    Region       int64 8B 229
    Region_name  <U52 208B 'United Kingdom of Great Britain and Northern Irel...
    Year         int64 8B 2020
    Item_name    (Item) <U31 6kB 'Apples and products' ... 'Poultry Meat'
    Item_group   (Item) <U24 5kB 'Fruits - Excluding Wine' ... 'Meat'
    Item_origin  (Item) <U16 3kB 'Vegetal Products' ... 'Animal Products'

In [5]:
# read csv file skipping first two lines
dataset_csv = pd.read_csv(
    'data/LCA+Meta-Analysis+of+Food+Products+-+Full+Model+v0+(1) - Database.csv',
    skiprows=2,
    dtype=str
    )


In [6]:
country_data = pd.read_csv(
    'data/LCA+Meta-Analysis+of+Food+Products+-+Full+Model+v0+(1) - Standardisation.csv',
    dtype=str,
    skiprows=2938,
    nrows=3146-2938,
    header=0,
    usecols=['FAO Country', 'Region']
    )

# First should be Afghanistan, last should be Zimbabwe
print(country_data.head())
print(country_data.tail())

      FAO Country         Region
0     Afghanistan  Mainland Asia
1         Albania         Europe
2         Algeria         Africa
3  American Samoa        Oceania
4          Angola         Africa
                   FAO Country         Region
203  Wallis and Futuna Islands        Oceania
204             Western Sahara         Africa
205                      Yemen  Mainland Asia
206                     Zambia         Africa
207                   Zimbabwe         Africa


In [7]:
european_countries = country_data[country_data['Region'] == 'Europe']['FAO Country'].tolist()
vprint(european_countries)
# remove the countries that are not in the list
dataset_csv = dataset_csv[dataset_csv['Country'].isin(european_countries)]

Albania
Austria
Belarus
Belgium
Bosnia and Herzegovina
Bulgaria
Croatia
Cyprus
Czech Republic
Denmark
Estonia
Faroe Islands
Finland
France
Germany
Greece
Hungary
Iceland
Ireland
Italy
Latvia
Lithuania
Luxembourg
Malta
Montenegro
Netherlands
Norway
Poland
Portugal
Republic of Moldova
Romania
Russian Federation
Serbia
Slovakia
Slovenia
Spain
Sweden
Switzerland
The former Yugoslav Republic of Macedonia
Ukraine
United Kingdom


In [8]:
item_list = dataset_csv['ID_LOSS'].unique().tolist()
vprint(item_list)

Wheat
Rye
Maize (Meal)
Barley (Beer)
Oats (Oatmeal)
Rice
Potatoes
Sugar beet
Beans & Pulses
Peas
Nuts
Soybeans (Soymilk)
Soybeans (Tofu)
Soybeans (Oil)
Sunflower (Oil)
Rapeseed (Oil)
Olives (Oil)
Tomatoes
Onions and Leeks
Root Vegetables
Cabbages and Other Brassicas
Other Vegetables
Citrus Fruit
Apples
Berries
Wine Grapes (Wine)
Other Fruit
Bovine Meat
Mutton & Goat Meat
Pig Meat
Poultry Meat
Milk
Cheese
Eggs
Fish (farmed)


## Calculate land use factors on a per item basis

In [9]:
# Plant items
for item in plant_items:

    # Get PN18 item name from matching dataframe
    pn18_item = matching_df_plants[matching_df_plants[cols[0]] == item][cols[3]].values[0]
    print(f"{item} ({pn18_item})", end=': ')

    # Get the land use factor for this item from the dataset
    plant_land_data = dataset_csv[dataset_csv['ID_LOSS'] == pn18_item][['Seed', 'Arable', 'Fallow', 'Loss']]

    plant_land_data = dataset_csv.loc[
        dataset_csv["ID_LOSS"] == pn18_item,
        ["Seed", "Arable", "Fallow", "Loss"]
    ].apply(pd.to_numeric, errors="coerce")

    avg_total_across_rows = plant_land_data.sum(axis=1, min_count=1).mean()
    print(avg_total_across_rows)

2617 (Apples): 0.5156857142857143
2513 (Oats (Oatmeal)): 4.236363636363635
2656 (Barley (Beer)): 0.458774647887324
2658 (Wine Grapes (Wine)): 2.263136851851852
2520 (nan): nan
2614 (Citrus Fruit): 0.4254214285714286
2625 (Other Fruit): 1.262268666666667
2620 (Berries): 0.9280912162035649
2745 (nan): nan
2582 (Rapeseed (Oil)): 9.19190566037736
2516 (Oats (Oatmeal)): 4.236363636363635
2586 (Rapeseed (Oil)): 9.19190566037736
2570 (Rapeseed (Oil)): 9.19190566037736
2602 (Onions and Leeks): 0.4391215454545454
2547 (Peas): 4.017857142857143
2531 (Potatoes): 0.652725
2549 (Beans & Pulses): 4.171538461538462
2574 (Rapeseed (Oil)): 9.19190566037736
2558 (Rapeseed (Oil)): 9.19190566037736
2515 (Oats (Oatmeal)): 4.236363636363635
2571 (Soybeans (Oil)): 8.05
2542 (Sugar Cane): nan
2537 (Sugar Beet): nan
2543 (nan): nan
2601 (Tomatoes): 0.08158503999999998
2605 (Other Vegetables): 0.6423817725000001
2511 (Wheat/Rye (Bread)): nan
2655 (Wine Grapes (Wine)): 2.263136851851852


In [10]:
# Animal items
for item in animal_items:

    # Get PN18 item name from matching dataframe
    pn18_item = matching_df_animals[matching_df_animals[cols[0]] == item][cols[3]].values[0]
    print(f"{item} ({pn18_item})", end=': ')

    # Get the land use factor for this item from the dataset
    animal_land_data = dataset_csv[dataset_csv['ID_LOSS'] == pn18_item][["Temp Past", "Perm Past"]]

    animal_land_data = dataset_csv.loc[
        dataset_csv["ID_LOSS"] == pn18_item,
        ["Temp Past", "Perm Past"]
    ].apply(pd.to_numeric, errors="coerce")

    avg_total_across_rows = animal_land_data.sum(axis=1, min_count=1).mean()
    print(avg_total_across_rows)

2731 (Beef): nan
2740 (nan): nan
2766 (Crustaceans (farmed)): nan
2743 (Milk): 1.3301204819277108
2765 (Crustaceans (farmed)): nan
2762 (Fish (farmed)): 0.0
2949 (Eggs): 0.0
2737 (nan): nan
2781 (Fish (farmed)): 0.0
2782 (Fish (farmed)): 0.0
2761 (Fish (farmed)): 0.0
2735 (Pig Meat): 7.765079365079365
2948 (Milk): 1.3301204819277108
2767 (Crustaceans (farmed)): nan
2732 (Mutton & Goat Meat): 134.81666666666666
2736 (Pig Meat): 7.765079365079365
2763 (Fish (farmed)): 0.0
2733 (Pig Meat): 7.765079365079365
2734 (Poultry Meat): 0.0
